In [ ]:
!pip install google-play-scraper pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.8 MB/s eta 0:00:00


In [ ]:
"""
Google Play Store Review Scraper untuk Livin' by Mandiri
Requirement: pip install google-play-scraper pandas openpyxl
"""

from google_play_scraper import app, Sort, reviews_all
import pandas as pd
from datetime import datetime
import time

# ============================================
# KONFIGURASI
# ============================================
APP_ID = 'id.bmri.livin'  # Package name Livin' by Mandiri
OUTPUT_FILE = 'livin_reviews_analysis.xlsx'
SCRAPE_COUNT = 2000  # Jumlah review yang ingin di-scrape (lebih banyak untuk filter tanggal)

# FILTER TANGGAL
START_DATE = '2025-06-14'  # Tanggal mulai (YYYY-MM-DD)
END_DATE = '2025-08-17'    # Tanggal akhir (YYYY-MM-DD)
USE_DATE_FILTER = True      # Set False jika tidak ingin filter tanggal

print("="*60)
print("GOOGLE PLAY STORE SCRAPER - LIVIN' BY MANDIRI")
print("="*60)

if USE_DATE_FILTER:
    print(f"Filter Tanggal: {START_DATE} hingga {END_DATE}")
else:
    print("Filter Tanggal: Tidak aktif (semua review)")

# ============================================
# STEP 1: SCRAPE INFORMASI APLIKASI
# ============================================
print("\n[1/4] Mengambil informasi aplikasi...")
try:
    app_info = app(APP_ID, lang='id', country='id')

    print(f"✓ Nama Aplikasi: {app_info['title']}")
    print(f"✓ Rating: {app_info['score']:.2f}/5.0")
    print(f"✓ Total Review: {app_info['reviews']:,}")
    print(f"✓ Total Download: {app_info['installs']}")
    print(f"✓ Versi: {app_info['version']}")
    print(f"✓ Update Terakhir: {app_info['updated']}")

    # Simpan info aplikasi
    app_summary = {
        'Nama Aplikasi': app_info['title'],
        'Package ID': APP_ID,
        'Rating Keseluruhan': app_info['score'],
        'Total Reviews': app_info['reviews'],
        'Total Download': app_info['installs'],
        'Versi Terbaru': app_info['version'],
        'Update Terakhir': app_info['updated'],
        'Tanggal Scraping': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'Filter Tanggal': f"{START_DATE} s/d {END_DATE}" if USE_DATE_FILTER else "Tidak aktif",
        'Jumlah Review Dianalisis': 'Akan dihitung setelah filtering'
    }

except Exception as e:
    print(f"✗ Error mengambil info aplikasi: {e}")
    exit()

# ============================================
# STEP 2: SCRAPE REVIEWS
# ============================================
print(f"\n[2/4] Mengambil {SCRAPE_COUNT} reviews...")
print("(Proses ini memakan waktu 2-5 menit, mohon tunggu...)")

try:
    # Scrape all reviews (sorted by newest)
    result = reviews_all(
        APP_ID,
        sleep_milliseconds=0,  # Delay antar request
        lang='id',
        country='id',
        sort=Sort.NEWEST,  # Urutkan dari yang terbaru
        count=SCRAPE_COUNT
    )

    print(f"✓ Berhasil mengambil {len(result)} reviews")

except Exception as e:
    print(f"✗ Error saat scraping: {e}")
    exit()

# ============================================
# STEP 3: PROSES DATA
# ============================================
print("\n[3/4] Memproses data reviews...")

# Konversi ke DataFrame
df = pd.DataFrame(result)

# Pilih kolom yang relevan
df_clean = df[[
    'userName',
    'score',
    'at',
    'content',
    'thumbsUpCount',
    'reviewCreatedVersion'
]].copy()

# Rename kolom
df_clean.columns = [
    'Nama_User',
    'Rating',
    'Tanggal',
    'Review_Text',
    'Thumbs_Up',
    'Versi_App'
]

# Format tanggal
df_clean['Tanggal'] = pd.to_datetime(df_clean['Tanggal'])

# FILTER BERDASARKAN RENTANG TANGGAL (jika aktif)
if USE_DATE_FILTER:
    start_date = pd.to_datetime(START_DATE)
    end_date = pd.to_datetime(END_DATE)

    total_before = len(df_clean)
    df_clean = df_clean[(df_clean['Tanggal'] >= start_date) & (df_clean['Tanggal'] <= end_date)]
    total_after = len(df_clean)

    print(f"✓ Filter tanggal diterapkan: {total_before} → {total_after} reviews")

    if total_after == 0:
        print("✗ PERINGATAN: Tidak ada review dalam rentang tanggal yang ditentukan!")
        print(f"   Coba perbesar SCRAPE_COUNT atau ubah rentang tanggal.")
        exit()

# Format tanggal untuk display
df_clean['Tanggal'] = df_clean['Tanggal'].dt.strftime('%Y-%m-%d')

# Tambah kolom kategori rating
def categorize_rating(score):
    if score <= 2:
        return 'Negatif (1-2★)'
    elif score == 3:
        return 'Netral (3★)'
    else:
        return 'Positif (4-5★)'

df_clean['Kategori'] = df_clean['Rating'].apply(categorize_rating)

# Tambah kolom panjang review
df_clean['Panjang_Review'] = df_clean['Review_Text'].str.len()

# Deteksi kata kunci keluhan
keywords = {
    'error': ['error', 'eror', 'gagal'],
    'lambat': ['lambat', 'lemot', 'lelet', 'lag'],
    'login': ['login', 'log in', 'masuk'],
    'crash': ['crash', 'force close', 'tutup sendiri', 'keluar sendiri'],
    'maintenance': ['maintenance', 'perbaikan', 'gangguan'],
    'customer_service': ['cs', 'customer service', 'layanan', 'komplain']
}

for category, words in keywords.items():
    pattern = '|'.join(words)
    df_clean[f'Mention_{category}'] = df_clean['Review_Text'].str.lower().str.contains(pattern, na=False)

print(f"✓ Data berhasil diproses")

# ============================================
# STEP 4: ANALISIS & STATISTIK
# ============================================
print("\n[4/4] Membuat analisis statistik...")

# Statistik umum
stats = {
    'Total Reviews': len(df_clean),
    'Rating Rata-rata': df_clean['Rating'].mean(),
    'Review Negatif (1-2★)': len(df_clean[df_clean['Rating'] <= 2]),
    'Review Netral (3★)': len(df_clean[df_clean['Rating'] == 3]),
    'Review Positif (4-5★)': len(df_clean[df_clean['Rating'] >= 4]),
    '% Review Negatif': f"{(len(df_clean[df_clean['Rating'] <= 2]) / len(df_clean) * 100):.1f}%",
    '% Review Netral': f"{(len(df_clean[df_clean['Rating'] == 3]) / len(df_clean) * 100):.1f}%",
    '% Review Positif': f"{(len(df_clean[df_clean['Rating'] >= 4]) / len(df_clean) * 100):.1f}%",
}

# Analisis kata kunci (hanya dari review negatif)
df_negative = df_clean[df_clean['Rating'] <= 2]
keyword_stats = {}
for category in keywords.keys():
    count = df_negative[f'Mention_{category}'].sum()
    pct = (count / len(df_negative) * 100) if len(df_negative) > 0 else 0
    keyword_stats[category] = {
        'Jumlah': count,
        'Persentase': f"{pct:.1f}%"
    }

# Distribusi rating per bulan (3 bulan terakhir)
df_clean['Bulan'] = pd.to_datetime(df_clean['Tanggal']).dt.to_period('M')
rating_by_month = df_clean.groupby(['Bulan', 'Kategori']).size().unstack(fill_value=0)

# ============================================
# STEP 5: EXPORT KE EXCEL
# ============================================
print(f"\n[5/5] Menyimpan hasil ke {OUTPUT_FILE}...")

with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    # Sheet 1: Info Aplikasi
    pd.DataFrame([app_summary]).T.to_excel(writer, sheet_name='Info_Aplikasi', header=False)

    # Sheet 2: Statistik Umum
    pd.DataFrame([stats]).T.to_excel(writer, sheet_name='Statistik_Umum', header=False)

    # Sheet 3: Analisis Kata Kunci
    pd.DataFrame(keyword_stats).T.to_excel(writer, sheet_name='Analisis_Kata_Kunci')

    # Sheet 4: Distribusi Rating per Bulan
    rating_by_month.to_excel(writer, sheet_name='Rating_per_Bulan')

    # Sheet 5: Semua Reviews
    df_clean.to_excel(writer, sheet_name='All_Reviews', index=False)

    # Sheet 6: Review Negatif Only (untuk analisis mendalam)
    df_negative.to_excel(writer, sheet_name='Review_Negatif', index=False)

    # Sheet 7: Top Reviews (most helpful)
    df_top = df_clean.nlargest(50, 'Thumbs_Up')
    df_top.to_excel(writer, sheet_name='Top_50_Reviews', index=False)

print(f"✓ File berhasil disimpan: {OUTPUT_FILE}")

# ============================================
# RINGKASAN HASIL
# ============================================
print("\n" + "="*60)
print("RINGKASAN HASIL SCRAPING")
print("="*60)
if USE_DATE_FILTER:
    print(f"Periode: {START_DATE} hingga {END_DATE}")
print(f"Total reviews yang di-scrape: {len(df_clean)}")
print(f"Rating rata-rata: {stats['Rating Rata-rata']:.2f}/5.0")
print(f"\nDistribusi Rating:")
print(f"  • Negatif (1-2★): {stats['Review Negatif (1-2★)']} ({stats['% Review Negatif']})")
print(f"  • Netral (3★): {stats['Review Netral (3★)']} ({stats['% Review Netral']})")
print(f"  • Positif (4-5★): {stats['Review Positif (4-5★)']} ({stats['% Review Positif']})")

print(f"\nKata Kunci dalam Review Negatif:")
for key, value in keyword_stats.items():
    print(f"  • {key.replace('_', ' ').title()}: {value['Jumlah']} reviews ({value['Persentase']})")

print(f"\n✓ Semua data tersimpan di: {OUTPUT_FILE}")
print("="*60)

# ============================================
# BONUS: EXPORT SAMPLE UNTUK PAPER
# ============================================
print("\n[BONUS] Membuat contoh tabel untuk paper...")

# Contoh 10 review negatif dengan thumbs up tertinggi (paling representatif)
sample_negative = df_negative.nlargest(10, 'Thumbs_Up')[
    ['Tanggal', 'Rating', 'Review_Text', 'Thumbs_Up', 'Versi_App']
]

print("\nContoh Review Negatif Paling Representatif:")
print("-" * 60)
for idx, row in sample_negative.head(3).iterrows():
    print(f"Rating: {row['Rating']}★ | Thumbs Up: {row['Thumbs_Up']} | Tanggal: {row['Tanggal']}")
    print(f"Review: {row['Review_Text'][:150]}...")
    print("-" * 60)

print("\n✓ Selesai! Silakan buka file Excel untuk analisis lengkap.")

GOOGLE PLAY STORE SCRAPER - LIVIN' BY MANDIRI
Filter Tanggal: 2025-06-14 hingga 2025-08-17

[1/4] Mengambil informasi aplikasi...
✓ Nama Aplikasi: Livin' by Mandiri
✓ Rating: 4.10/5.0
✓ Total Review: 292,096
✓ Total Download: 10.000.000+
✓ Versi: 3.1.0
✓ Update Terakhir: 1755617450

[2/4] Mengambil 2000 reviews...
(Proses ini memakan waktu 2-5 menit, mohon tunggu...)
✓ Berhasil mengambil 274441 reviews

[3/4] Memproses data reviews...
✓ Filter tanggal diterapkan: 274441 → 8231 reviews
✓ Data berhasil diproses

[4/4] Membuat analisis statistik...

[5/5] Menyimpan hasil ke livin_reviews_analysis.xlsx...
✓ File berhasil disimpan: livin_reviews_analysis.xlsx

RINGKASAN HASIL SCRAPING
Periode: 2025-06-14 hingga 2025-08-17
Total reviews yang di-scrape: 8231
Rating rata-rata: 3.58/5.0

Distribusi Rating:
  • Negatif (1-2★): 2681 (32.6%)
  • Netral (3★): 494 (6.0%)
  • Positif (4-5★): 5056 (61.4%)

Kata Kunci dalam Review Negatif:
  • Error: 280 reviews (10.4%)
  • Lambat: 357 reviews (13.3%)
